In [ ]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [5]:
df = pd.read_csv('car-details.csv')
df.sample(5)

,name,company,model,edition,year,owner,fuel,seller_type,transmission,km_driven,mileage_mpg,engine_cc,max_power_bhp,torque_nm,seats,selling_price
1334,Mahindra Bolero DI DX 7 Seater,Mahindra,Bolero,DI DX 7 Seater,2009,Second,Diesel,Individual,Manual,150000,31.95,2523.0,63.00,180.0,7.0,270000
5827,Chevrolet Enjoy 1.3 TCDi LS 7,Chevrolet,Enjoy,1.3 TCDi LS 7,2016,Second,Diesel,Individual,Manual,110000,42.78,1248.0,73.74,172.5,7.0,280000
5781,Maruti Swift VDI,Maruti,Swift,VDI,2012,Second,Diesel,Individual,Manual,56000,53.80,1248.0,74.00,190.0,5.0,425000
5837,Maruti Alto 800 LXI,Maruti,Alto,800 LXI,2017,First,Petrol,Individual,Manual,27000,58.03,796.0,47.30,69.0,5.0,300000
5663,Datsun GO Plus A,Datsun,GO,Plus A,2016,First,Petrol,Individual,Manual,20000,48.47,1198.0,67.00,104.0,7.0,385000


In [6]:
df = df.drop(columns=['name', 'model', 'edition'])

In [9]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.drop_duplicates(inplace=True)

In [10]:
X= df.drop(columns=['selling_price'])
y = df.selling_price.copy()

(6907, 12) (6907,)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(5525, 12) (5525,)
(1382, 12) (1382,)


In [17]:
num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = [col for col in X_train.columns if col not in num_cols]

print(num_cols)
print()
print(cat_cols)

['year', 'km_driven', 'mileage_mpg', 'engine_cc', 'max_power_bhp', 'torque_nm', 'seats']

['company', 'owner', 'fuel', 'seller_type', 'transmission']


In [19]:
num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),  
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols),    
    ('cat', cat_pipe, cat_cols)
])
 

In [20]:
regressor = RandomForestRegressor(n_estimators=10, max_depth =5,random_state=42)

In [22]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', regressor)
])

rf_model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['year', 'km_driven',
                                                   'mileage_mpg', 'engine_cc',
                                                   'max_power_bhp', 'torque_nm',
                                                   'seats']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value='missing',
                                                                                 strategy='constant')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['company', 'owner', 'fuel',
                                                   'seller_type',
                                                   'transmission'])])),
                ('model',
                 RandomForestRegressor(max_depth=5, n_estimators=10,
                                       random_state=42))])

In [26]:
y_train_pred  = rf_model.predict(X_train)
train_rmse = root_mean_squared_error(y_train, y_train_pred)
print(f"Train RMSE: {train_rmse:.2f}")

y_test_pred = rf_model.predict(X_test) 
test_rmse = root_mean_squared_error(y_test, y_test_pred)
print(f"Test RMSE: {test_rmse:.2f}")

Train RMSE: 169947.49
Test RMSE: 172392.13
